In [ ]:
# configuration class for mouseReMoCo application

from dataclasses import dataclass, field
from typing import Optional, Tuple
from enum import Enum


class TaskType(Enum):
    """Task types supported by the application"""

    CIRCULAR = "circular"
    LINEAR = "linear"


@dataclass
class Configuration:
    """
    Main configuration class mirroring Java Configuration.java
    Handles all application settings including:
    - Screen and window configuration
    - Circular and linear task parameters
    - Visual styling (colors, cursors, fonts)
    - Input and output settings
    """

    # ===== Window Configuration =====
    title: str = "Wacom Tablet Test"
    target_monitor: int = 2
    width: int = None
    height: int = None
    margin_multiplier: int = 5

    # ===== Screen & Window Configuration =====
    screen_width: int = 0
    screen_height: int = 0
    drawable_width: int = 0
    drawable_height: int = 0
    frame_location_x: int = 0
    frame_location_y: int = 0
    frame_insets: dict = field(
        default_factory=lambda: {"top": 0, "bottom": 0, "left": 0, "right": 0}
    )
    frame_undecorated: bool = False
    used_screen_id: int = 0

    # ===== Circular Task Parameters =====
    task_string: str = "circular"
    center_x: int = 0
    center_y: int = 0
    corner_x: int = 0
    corner_y: int = 0
    external_radius: int = 150
    internal_radius: int = 80
    border_radius: int = 1
    circle_perimeter_mm: int = 0

    # Circular task derived values
    task_radius: float = 0.0
    tolerance_px: int = 0
    index_of_difficulty: float = 0.0
    internal_limit: int = 0
    external_limit: int = 0

    # ===== Linear Task Parameters =====
    inter_line_distance_mm: int = 150
    line_height_mm: int = 100
    mm2px: float = 0.0

    # ===== Auditory Rhythm =====
    half_period: int = 2000

    # ===== Cursor Configuration =====
    cursor_radius: int = 16
    cursor_color_record: Tuple[int, int, int] = (255, 0, 0)  # RGB red
    cursor_color_wait: Tuple[int, int, int] = (255, 255, 0)  # RGB yellow

    # ===== Visual Styling =====
    border_color: Tuple[int, int, int] = (255, 255, 255)  # RGB white
    background_color: Tuple[int, int, int] = (0, 0, 0)  # RGB black
    text_color: Tuple[int, int, int] = (255, 255, 255)  # RGB white

    # ===== Sequence Configuration =====
    auto_start: int = 3600  # seconds before auto start
    cycle_max_number: int = 6  # Move-Rest cycle number
    cycle_duration: int = 20  # seconds for a Move or Rest (half-cycle)
    is_target_hidden_during_pause: bool = False

    # ===== Font Configuration =====
    font_size: int = 20
    font_family: str = "Courier"

    # ===== Flags =====
    is_with_lsl: bool = False  # Lab Streaming Layer
    is_with_pause_target: bool = False

    # ===== Application State =====
    step: str = ""

    def __post_init__(self):
        """Initialize derived values after dataclass initialization"""
        self._update_circular_task()

    def _update_circular_task(self):
        """Update circular task derived values"""
        if self.task_string == "circular":
            # Limits of the path
            self.internal_limit = self.internal_radius + self.cursor_radius
            self.external_limit = (
                self.external_radius - self.cursor_radius - self.border_radius
            )

            # ID in the steering law (Accot & Zhai 1999)
            self.task_radius = (self.internal_limit + self.external_limit) / 2.0
            self.tolerance_px = self.external_limit - self.internal_limit

            if self.tolerance_px > 0:
                self.index_of_difficulty = (
                    2.0 * 3.14159 * self.task_radius
                ) / self.tolerance_px

    def set_index_of_difficulty(self, index_of_difficulty: float):
        """Set index of difficulty and adjust circle parameters"""
        if self.task_string != "circular":
            return

        # Calculate new tolerance width
        w = (3.14159 * self.external_limit) / (index_of_difficulty + 3.14159)
        wn = round(2 * w)

        # Update internal limit and radius
        self.internal_limit = self.external_limit - wn
        self.internal_radius = self.internal_limit - self.cursor_radius

        # Recalculate derived values
        self._update_circular_task()

    def set_circular_task(self):
        """Initialize circular task parameters"""
        self._update_circular_task()

    def set_linear_task(self):
        """Initialize linear task parameters"""
        # Linear task setup would go here
        pass

    def set_circle_perimeter(self, perimeter_mm: int, screen_resolution_ppi: float):
        """Set circle perimeter and adjust circle parameters accordingly"""
        if perimeter_mm <= 0 or screen_resolution_ppi <= 0:
            return

        # Convert mm to pixels
        self.circle_perimeter_mm = perimeter_mm
        perimeter_px = perimeter_mm * screen_resolution_ppi / 25.4  # 25.4 mm per inch

        # Calculate new radius and tolerance
        self.task_radius = perimeter_px / (2.0 * 3.14159)
        tolerance = perimeter_px / self.index_of_difficulty

        external_limit = self.task_radius + tolerance / 2.0
        internal_limit = self.task_radius - tolerance / 2.0

        external_radius = external_limit + self.cursor_radius + self.border_radius
        internal_radius = internal_limit - self.cursor_radius

        self.external_radius = round(external_radius)
        self.internal_radius = round(internal_radius)

        self.corner_x = self.drawable_width // 2 - self.external_radius
        self.corner_y = self.drawable_height // 2 - self.external_radius

        self._update_circular_task()

    def calculate_radii(self, screen_width: int, screen_height: int) -> tuple[int, int]:
        """Calculate circle radii based on screen dimensions and margin settings"""
        # Calculate available space accounting for margins
        margin_px = self.margin_multiplier * self.cursor_radius
        available_width = screen_width - 2 * margin_px
        available_height = screen_height - 2 * margin_px

        # Use smaller dimension to ensure circle fits
        max_diameter = min(available_width, available_height)

        if max_diameter <= 0:
            return self.external_radius, self.internal_radius

        # External radius is half the maximum diameter
        external_radius = max_diameter // 2

        # Internal radius is 60% of external radius (creates 40% wide tolerance band)
        internal_radius = int(external_radius * 0.6)

        return external_radius, internal_radius

    def set_center_x(self, center_x: int):
        """Set center X and update corner X accordingly"""
        self.center_x = center_x
        self.corner_x = center_x - self.external_radius

    def set_center_y(self, center_y: int):
        """Set center Y and update corner Y accordingly"""
        self.center_y = center_y
        self.corner_y = center_y - self.external_radius

    def set_corner_x(self, corner_x: int):
        """Set corner X and update center X accordingly"""
        self.corner_x = corner_x
        self.center_x = corner_x + self.external_radius

    def set_corner_y(self, corner_y: int):
        """Set corner Y and update center Y accordingly"""
        self.corner_y = corner_y
        self.center_y = corner_y + self.external_radius

    def to_string(self) -> str:
        """Generate configuration string representation"""
        parts = [
            f"software mouseReMoCo",
            f"isWithLSL {self.is_with_lsl}",
            f"screenWidth {self.screen_width}",
            f"screenHeight {self.screen_height}",
            f"centerX {self.center_x}",
            f"centerY {self.center_y}",
            f"autoStart {self.auto_start}",
            f"cycleMaxNumber {self.cycle_max_number}",
            f"cycleDuration {self.cycle_duration}",
            f"borderColor {self.border_color}",
            f"backgroundColor {self.background_color}",
            f"textColor {self.text_color}",
            f"task {self.task_string}",
        ]

        if self.task_string == "circular":
            parts.extend(
                [
                    f"cornerX {self.corner_x}",
                    f"cornerY {self.corner_y}",
                    f"externalRadius {self.external_radius}",
                    f"internalRadius {self.internal_radius}",
                    f"borderRadius {self.border_radius}",
                    f"cursorRadius {self.cursor_radius}",
                    f"indexOfDifficulty {self.index_of_difficulty:.2f}",
                    f"taskRadius {self.task_radius:.2f}",
                    f"taskTolerance {self.tolerance_px}",
                ]
            )

        return ";".join(parts)

In [ ]:
# Screen management — ScreenInfo + ScreenManager utilities
from dataclasses import dataclass

from PyQt6.QtWidgets import QApplication, QWidget


@dataclass
class ScreenInfo:
    """Information about a screen"""

    name: str
    index: int
    width: int
    height: int
    pos_x: int
    pos_y: int
    phys_width_mm: float
    phys_height_mm: float
    dpi_x: float
    dpi_y: float
    dpi_avg: float
    diag_inches: float


class ScreenManager:
    """Static utility methods for screen and window management"""

    @staticmethod
    def get_screen_info(screen, app: QApplication) -> ScreenInfo:
        """Extract detailed info from a QScreen object"""
        geometry = screen.geometry()
        phys_size = screen.physicalSize()

        # Calculate DPI
        dpi_x = geometry.width() / (phys_size.width() / 25.4)
        dpi_y = geometry.height() / (phys_size.height() / 25.4)
        dpi_avg = (dpi_x + dpi_y) / 2

        # Calculate diagonal in inches
        diag_inches = (phys_size.width() ** 2 + phys_size.height() ** 2) ** 0.5 / 25.4

        return ScreenInfo(
            name=screen.name(),
            index=app.screens().index(screen),
            width=geometry.width(),
            height=geometry.height(),
            pos_x=geometry.x(),
            pos_y=geometry.y(),
            phys_width_mm=phys_size.width(),
            phys_height_mm=phys_size.height(),
            dpi_x=dpi_x,
            dpi_y=dpi_y,
            dpi_avg=dpi_avg,
            diag_inches=diag_inches,
        )

    @staticmethod
    def get_all_screens(app: QApplication) -> list[ScreenInfo]:
        """Get info for all connected screens"""
        return [ScreenManager.get_screen_info(screen, app) for screen in app.screens()]

    @staticmethod
    def print_all_screens(screens: list[ScreenInfo]):
        """Print formatted screen information"""
        print("=" * 60)
        print("Available Screens:")
        print("=" * 60)
        for s in screens:
            print(f"\nScreen {s.index + 1}: {s.name}")
            print(f"  Geometry: {s.width}×{s.height} @ ({s.pos_x}, {s.pos_y})")
            print(f"  DPI: {s.dpi_x:.1f}×{s.dpi_y:.1f} (avg: {s.dpi_avg:.1f})")
            print(f"  Physical: {s.phys_width_mm:.1f}×{s.phys_height_mm:.1f} mm")
            print(f'  Diagonal: {s.diag_inches:.1f}"')

    @staticmethod
    def get_target_screen(
        screens: list[ScreenInfo], config: Configuration
    ) -> ScreenInfo:
        """Get the target screen with safe fallback"""
        target_index = config.target_monitor - 1  # Convert 1-indexed to 0-indexed
        if 0 <= target_index < len(screens):
            return screens[target_index]
        print(f"⚠ Monitor {config.target_monitor} not found, using primary screen")
        return screens[0]

    @staticmethod
    def get_usable_screen_size(
        app: QApplication, screen_info: ScreenInfo
    ) -> tuple[int, int]:
        """Get usable screen size (excludes taskbars, etc.)"""
        screen = app.screens()[screen_info.index]
        usable = screen.availableGeometry()
        return usable.width(), usable.height()

    @staticmethod
    def get_window_drawable_area(
        widget: QWidget, initial_width: int, initial_height: int
    ) -> tuple[int, int, dict]:
        """Calculate actual drawable area accounting for window frame insets"""
        frame_geometry = widget.frameGeometry()
        content_geometry = widget.geometry()

        # Calculate frame insets
        insets = {
            "top": content_geometry.top() - frame_geometry.top(),
            "bottom": frame_geometry.bottom() - content_geometry.bottom(),
            "left": content_geometry.left() - frame_geometry.left(),
            "right": frame_geometry.right() - content_geometry.right(),
        }

        # Calculate actual drawable dimensions
        actual_width = initial_width - insets["left"] - insets["right"]
        actual_height = initial_height - insets["top"] - insets["bottom"]

        return actual_width, actual_height, insets

In [ ]:
# Cursor Factory — Generate custom cursor images

from PyQt6.QtGui import QPixmap, QPainter, QColor, QCursor
from PyQt6.QtCore import Qt, QPoint


class CursorFactory:
    """Factory for creating custom cursor images with filled circles and crosshairs"""

    @staticmethod
    def create_cursor(
        radius: int, 
        color: tuple[int, int, int], 
        background_color: tuple[int, int, int] = (0, 0, 0)
    ) -> QCursor:
        """
        Create a custom cursor with a filled circle and center crosshair.
        
        Args:
            radius: Cursor circle radius in pixels
            color: RGB tuple (r, g, b) for circle color
            background_color: RGB tuple for background (for crosshair visibility)
        
        Returns:
            QCursor with the custom cursor image
        """
        diameter = radius * 2
        
        # Create transparent pixmap
        pixmap = QPixmap(diameter, diameter)
        pixmap.fill(Qt.GlobalColor.transparent)
        
        # Create painter and draw on pixmap
        painter = QPainter(pixmap)
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)
        
        # Draw filled circle
        circle_color = QColor(*color)
        painter.fillEllipse(0, 0, diameter, diameter, circle_color)
        
        # Draw center crosshair (two perpendicular lines)
        crosshair_color = QColor(*background_color)
        painter.setPen(crosshair_color)
        
        crosshair_length = 4  # pixels extending from center in each direction
        center = radius
        
        # Horizontal line
        painter.drawLine(
            center - crosshair_length, center,
            center + crosshair_length, center
        )
        
        # Vertical line
        painter.drawLine(
            center, center - crosshair_length,
            center, center + crosshair_length
        )
        
        painter.end()
        
        # Create cursor with hotspot at center
        hotspot = QPoint(radius, radius)
        cursor = QCursor(pixmap, hotspot.x(), hotspot.y())
        
        return cursor

    @staticmethod
    def create_record_cursor(config: "Configuration") -> QCursor:
        """Create cursor for recording state (red circle)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_record,
            background_color=config.background_color
        )

    @staticmethod
    def create_wait_cursor(config: "Configuration") -> QCursor:
        """Create cursor for waiting state (yellow circle)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_wait,
            background_color=config.background_color
        )

    @staticmethod
    def create_out_cursor(config: "Configuration") -> QCursor:
        """Create cursor for outside target state (darkened record color)"""
        # Darken the record color by reducing RGB values
        darkened = tuple(max(0, c // 2) for c in config.cursor_color_record)
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=darkened,
            background_color=config.background_color
        )

In [ ]:
# Circular target rendering

from dataclasses import dataclass

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QBrush, QColor, QPainter, QPen


@dataclass
class CircularTaskConfig:
    """Configuration for circular target task"""

    external_radius: int = 150  # pixels
    internal_radius: int = 80  # pixels
    background_color: str = "black"
    path_color: str = "#333333"  #  darkgray < "#333333"  < "#1a1a1a" < black
    circle_border_color: str = "white"
    circle_border_width: int = 2

    @staticmethod
    def rgb_to_hex(rgb_tuple: tuple[int, int, int]) -> str:
        """Convert RGB tuple (r, g, b) to hex color string"""
        r, g, b = rgb_tuple
        return f"#{r:02x}{g:02x}{b:02x}"


class CircularTargetWidget:
    """Draw circular target with tolerance band"""

    def __init__(self, config: CircularTaskConfig = None):
        self.config = config or CircularTaskConfig()

    def draw(self, painter: QPainter, center_x: int, center_y: int):
        """Draw the circular target"""
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)

        # Draw external circle (border)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.external_radius,
            fill_color=self.config.path_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

        # Draw internal circle (background)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.internal_radius,
            fill_color=self.config.background_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

    def _draw_filled_circle(
        self,
        painter: QPainter,
        x: int,
        y: int,
        radius: int,
        fill_color: str,
        border_color: str,
        border_width: int,
    ):
        """Helper to draw filled circle with border"""
        # Set fill color
        fill = QColor(fill_color)
        painter.setBrush(QBrush(fill))

        # Set border (pen)
        border = QColor(border_color)
        pen = QPen(border)
        pen.setWidth(border_width)
        painter.setPen(pen)

        # Draw circle
        painter.drawEllipse(x - radius, y - radius, 2 * radius, 2 * radius)

In [ ]:
# Window setup orchestration


class WindowSetup:
    """Encapsulates the complete window setup and initialization process"""

    def __init__(
        self,
        config: Configuration,
        app: QApplication,
        circle_config: "CircularTaskConfig",
        tablet_test_class: type,
    ):
        self.config = config
        self.app = app
        self.circle_config = circle_config
        self.tablet_test_class = tablet_test_class
        self.screens = None
        self.target_screen_info = None
        self.usable_width = None
        self.usable_height = None
        self.widget = None

    def initialize_screens(self):
        """Step 1: Detect screens and select target"""
        self.screens = ScreenManager.get_all_screens(self.app)
        ScreenManager.print_all_screens(self.screens)
        self.target_screen_info = ScreenManager.get_target_screen(
            self.screens, self.config
        )
        self.usable_width, self.usable_height = ScreenManager.get_usable_screen_size(
            self.app, self.target_screen_info
        )

    def calculate_initial_radii(self) -> tuple[int, int, "CircularTaskConfig"]:
        """Step 2-3: Calculate initial radii and create circle config"""
        external_radius, internal_radius = self.config.calculate_radii(
            screen_width=self.usable_width,
            screen_height=self.usable_height,
        )
        print(f"Cursor radius: {self.config.cursor_radius} px")
        print(
            f"Circle margin: {self.config.margin_multiplier} × {self.config.cursor_radius} = "
            f"{self.config.margin_multiplier * self.config.cursor_radius} px"
        )
        print(f"External radius: {external_radius} px")
        print(f"Internal radius: {internal_radius} px\n")

        circle_config = CircularTaskConfig(
            external_radius=external_radius,
            internal_radius=internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )
        return external_radius, internal_radius, circle_config

    def create_widget(self) -> QWidget:
        """Step 4: Create and position window widget"""
        widget = self.tablet_test_class(
            self.target_screen_info,
            self.circle_config,
            self.usable_width,
            self.usable_height,
            config=self.config,
        )
        widget.setWindowTitle(self.config.title)
        widget.move(self.target_screen_info.pos_x, self.target_screen_info.pos_y)

        window_width = self.config.width or self.usable_width
        window_height = self.config.height or self.usable_height
        widget.resize(window_width, window_height)

        return widget

    def measure_and_correct_dimensions(self):
        """Step 5-8: Measure frame insets and update widget with corrected dimensions"""
        # Show window to make frame insets calculable
        self.widget.show()
        self.app.processEvents()

        # Measure actual drawable area
        actual_width, actual_height, insets = ScreenManager.get_window_drawable_area(
            self.widget, self.usable_width, self.usable_height
        )
        print(
            f"\nWindow frame insets: Top={insets['top']}, Bottom={insets['bottom']}, Left={insets['left']}, Right={insets['right']}"
        )

        # Recalculate radii with actual drawable area
        external_radius, internal_radius = self.config.calculate_radii(
            actual_width, actual_height
        )

        # Create corrected circle config
        corrected_circle_config = CircularTaskConfig(
            external_radius=external_radius,
            internal_radius=internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )

        # Update widget with corrected values
        self.widget.circular_target = CircularTargetWidget(
            config=corrected_circle_config
        )
        self.widget.drawable_width = actual_width
        self.widget.drawable_height = actual_height
        self.widget.center_x = actual_width // 2
        self.widget.center_y = actual_height // 2
        self.widget.update()

    def finalize_display(self):
        """Step 9: Finalize window display"""
        self.widget.raise_()
        self.widget.activateWindow()
        self.widget.setFocus()
        print(f"{'='*60}\n")

    def create_and_display(self) -> QWidget:
        """Execute the complete setup pipeline"""
        self.initialize_screens()
        _, _, self.circle_config = self.calculate_initial_radii()
        self.widget = self.create_widget()
        self.measure_and_correct_dimensions()
        self.finalize_display()
        return self.widget

In [ ]:
# Application & Execution — TabletTest widget and main entry point

import sys

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QColor, QPainter, QPen, QTabletEvent
from PyQt6.QtWidgets import QApplication, QWidget


class TabletTest(QWidget):
    def __init__(
        self,
        screen_info,
        circle_config,
        usable_width: int,
        usable_height: int,
        actual_drawable_width: int = None,
        actual_drawable_height: int = None,
        config: "Configuration" = None,
    ):
        super().__init__()
        self.screen_info = screen_info
        self.config = config
        self.circular_target = CircularTargetWidget(config=circle_config)
        self.usable_width = usable_width
        self.usable_height = usable_height
        # Use actual drawable dimensions if provided, otherwise use usable dimensions
        self.drawable_width = actual_drawable_width or usable_width
        self.drawable_height = actual_drawable_height or usable_height
        self.center_x = usable_width // 2
        self.center_y = usable_height // 2

    def paintEvent(self, event):
        painter = QPainter(self)
        # Use background color from config, or default to black
        if self.config and self.config.background_color:
            bg_color = QColor(*self.config.background_color)
        else:
            bg_color = Qt.GlobalColor.black
        painter.fillRect(self.rect(), bg_color)

        # Draw green-yellow rectangles showing drawable screen limits
        shift = 0  # small shift to see the border more clearly (-1,suppresses the green rect)
        painter.setPen(QPen(Qt.GlobalColor.green, 1))
        painter.drawRect(
            shift,
            shift + 1,  # drawRect needs this correction (test on OSx)
            self.drawable_width - 2 * shift,
            self.drawable_height - 2 * shift - 1,
        )
        shift += 5
        painter.setPen(QPen(Qt.GlobalColor.yellow, 1))
        painter.drawRect(
            shift,
            shift + 1,
            self.drawable_width - 2 * (shift),
            self.drawable_height - 2 * (shift) - 1,
        )

        # Draw the circular target
        self.circular_target.draw(painter, self.center_x, self.center_y)

    def tabletEvent(self, event: QTabletEvent):
        """Print tablet coordinates, pressure, and tilt angles"""
        print(
            f"X: {event.position().x():.1f}, Y: {event.position().y():.1f}, "
            f"Pressure: {event.pressure():.2f}, Tilt X: {event.xTilt():.1f}°, "
            f"Tilt Y: {event.yTilt():.1f}°"
        )
        sys.stdout.flush()
        event.accept()
        print(f"Mouse: X: {event.position().x():.1f}, Y: {event.position().y():.1f}")
    def mouseMoveEvent(self, event):
        """Print mouse position"""
        print(f"Mouse: X: {event.position().x():.1f}, Y: {event.position().y():.1f}")
        sys.stdout.flush()

    def mousePressEvent(self, event):
        """Print mouse click position"""
        print(
            f"Mouse click at: X={event.position().x():.1f}, Y={event.position().y():.1f}"
        )
        sys.stdout.flush()


# ===== CONFIGURATION =====
config = Configuration(
    title="Wacom Pen Test - PyQt6",
    target_monitor=2,  # 1=primary, 2=second monitor, etc.
    cursor_radius=16,
    margin_multiplier=5,
    background_color=(50, 0, 0), 
)


# ===== SETUP =====
app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)

# Initialize window setup with placeholder circle config
setup = WindowSetup(
    config=config,
    app=app,
    circle_config=CircularTaskConfig(),  # Will be overwritten in setup
    tablet_test_class=TabletTest,
)


# Create and display the window (executes all 9 steps internally)
widget = setup.create_and_display()

# Run the Qt event loop
app.exec()
